In [ ]:
# 데이터 로드 및 탐색
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("data/hour.csv")
print("데이터 크기:", df.shape)
df.info()
print("결측치:")
print(df.isnull().sum())
print("기초 통계량:")
print(df.describe())

hour_avg = df.groupby("hr")["cnt"].mean()

print("시간대별 평균 대여량:")
plt.figure(figsize=(10, 4))
plt.plot(hour_avg.index, hour_avg.values, marker="o")
plt.title("Average Bike Rentals by Hour")
plt.xlabel("Hour")
plt.ylabel("Average Rentals")
plt.show()

print("대여량 분포:")
plt.figure(figsize=(8, 4))
plt.hist(df["cnt"], bins=20)
plt.title("Bike Rental Distribution")
plt.xlabel("Rental Count")
plt.ylabel("Frequency")
plt.show()

In [ ]:
# 데이터 전처리 및 원핫 인코딩
df_clean = df.drop(columns=[
    "instant", "dteday", "casual", "registered"
])
print("불필요한 컬럼 제거 후 데이터 확인")
print(df_clean.head())

df_encoded = pd.get_dummies(
    df_clean,
    columns=["season", "mnth", "hr", "weekday", "weathersit"],
    dtype=int
)
print("원핫 인코딩 후 데이터 크기 확인:")
print(df_encoded.shape)

X = df_encoded.drop(columns="cnt")
y = df_encoded["cnt"]

print("입력 데이터:", X.shape)
print("정답 데이터:", y.shape)

In [ ]:
# 모델 학습
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)

rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

prediction_df = pd.DataFrame({
    "Actual": y_test.values,
    "LinearRegression": lr_pred,
    "RandomForestRegressor": rf_pred
})
print("실제값 예측값 비교")
print(prediction_df.head(10))

In [ ]:
# 성능 평가 및 분석
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

lr_mae = mean_absolute_error(y_test, lr_pred)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_pred))
lr_r2 = r2_score(y_test, lr_pred)

rf_mae = mean_absolute_error(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_r2 = r2_score(y_test, rf_pred)

result_df = pd.DataFrame({
    "Model": ["LinearRegression", "RandomForestRegressor"],
    "MAE": [lr_mae, rf_mae],
    "RMSE": [lr_rmse, rf_rmse],
    "R2": [lr_r2, rf_r2]
})
print("모델 성능 비교:")
print(result_df.round(4))

print("실제값 vs 예측값:")
plt.figure(figsize=(10, 5))
plt.plot(y_test.values[:100], label="Actual")
plt.plot(lr_pred[:100], label="LinearRegression")
plt.plot(rf_pred[:100], label="RandomForest")
plt.title("Actual vs Predicted Bike Rentals")
plt.xlabel("Sample")
plt.ylabel("Rental Count")
plt.legend()
plt.show()

importance = pd.Series(
    rf_model.feature_importances_,
    index=X_train.columns
)

importance = importance.sort_values(ascending=False)
print("주요 변수 중요도:")
print(importance.head(10))

RandomForestRegressor는 LinearRegression보다 MAE와 RMSE가 낮고 R²가 높아 전반적으로 우수한 예측 성능을 보였다.

특히 RandomForestRegressor의 MAE는 약 55대로, LinearRegression의 약 99대보다 예측 오차가 작았다.

따라서 이번 공공자전거 대여량 예측에서는 RandomForestRegressor가 더 적합한 모델이다.

변수 중요도 분석 결과, 체감온도와 17~18시 시간대, 근무일 여부가 자전거 대여량 예측에 중요한 변수로 나타났다.